In [1]:
import numpy as np

class DecisionStump:
    def __init__(self):
        self.feature_index = None
        self.threshold = None
        self.polarity = 1
        self.alpha = None  # learner weight

    def predict(self, X):
        n_samples = X.shape[0]
        predictions = np.ones(n_samples)

        if self.polarity == 1:
            predictions[X[:, self.feature_index] < self.threshold] = -1
        else:
            predictions[X[:, self.feature_index] > self.threshold] = -1

        return predictions


In [3]:
class AdaBoost:
    def __init__(self, n_estimators=10):
        self.n_estimators = n_estimators
        self.stumps = []

    def fit(self, X, y):
        X = X.values
        n_samples, n_features = X.shape

        # Initialize weights uniformly
        w = np.ones(n_samples) / n_samples

        self.stumps = []

        for _ in range(self.n_estimators):
            stump = DecisionStump()
            min_error = float('inf')

            # Try all features and thresholds
            for feature_i in range(n_features):
                thresholds = np.unique(X[:, feature_i])
                print(thresholds)
                for threshold in thresholds:
                    for polarity in [1, -1]:
                        predictions = np.ones(n_samples) #Start by predicting +1 for all samples
                        if polarity == 1:
                            predictions[X[:, feature_i] < threshold] = -1
                        else:
                            predictions[X[:, feature_i] > threshold] = -1

                        # Calculate weighted error
                        error = np.sum(w[y != predictions])

                        if error < min_error:
                            min_error = error
                            stump.feature_index = feature_i
                            stump.threshold = threshold
                            stump.polarity = polarity

            # Compute learner weight (alpha)
            EPS = 1e-10
            stump.alpha = 0.5 * np.log((1 - min_error) / (min_error + EPS))

            # Update sample weights
            predictions = stump.predict(X)
            w *= np.exp(-stump.alpha * y * predictions)
            w /= np.sum(w)

            self.stumps.append(stump)

    def predict(self, X):
        X = X.values
        stump_preds = np.array([stump.alpha * stump.predict(X) for stump in self.stumps])
        y_pred = np.sign(np.sum(stump_preds, axis=0))
        return y_pred


In [4]:
import pandas as pd
import numpy as np

dataset = pd.read_csv("/content/Heart Attack.csv")
dataset = dataset[['impluse', 'pressurehight', 'pressurelow', 'glucose', 'kcm', 'troponin', 'class']]
dataset['class'] = dataset['class'].map({'positive': 1, 'negative': -1})


train = dataset.sample(frac=0.8, random_state=0)
test = dataset.drop(train.index) # remaining 20%

X_ori = train.iloc[:, :-1]
X_ori_mean = X_ori.mean(axis=0)
X_ori_std  = X_ori.std(axis=0)
X_norm = (X_ori - X_ori_mean) / X_ori_std
y_target = train.iloc[:, -1]

X_train = X_norm
y_train = y_target


X_test_ori = test.iloc[:, :-1]
y_test = test.iloc[:, -1]
# Normalize test features using TRAIN mean & std
X_test = (X_test_ori - X_ori_mean) / X_ori_std


model = AdaBoost(n_estimators=10)
model.fit(X_train, y_train)

[-1.03538635e+00 -7.56093297e-01 -6.86270032e-01 -5.98990952e-01
 -5.81535136e-01 -5.11711872e-01 -4.94256055e-01 -4.76800239e-01
 -4.59344423e-01 -4.41888607e-01 -4.24432791e-01 -4.06976975e-01
 -3.89521159e-01 -3.72065343e-01 -3.54609527e-01 -3.37153711e-01
 -3.19697894e-01 -3.02242078e-01 -2.84786262e-01 -2.67330446e-01
 -2.49874630e-01 -2.32418814e-01 -2.14962998e-01 -1.97507182e-01
 -1.80051366e-01 -1.62595550e-01 -1.45139733e-01 -1.27683917e-01
 -1.10228101e-01 -9.27722852e-02 -7.53164691e-02 -5.78606530e-02
 -4.04048369e-02 -2.29490208e-02 -5.49320469e-03  1.19626114e-02
  2.94184275e-02  4.68742436e-02  6.43300597e-02  8.17858758e-02
  9.92416919e-02  1.16697508e-01  1.34153324e-01  1.51609140e-01
  1.69064956e-01  1.86520772e-01  2.03976588e-01  2.21432405e-01
  2.38888221e-01  2.56344037e-01  2.73799853e-01  2.91255669e-01
  3.08711485e-01  3.26167301e-01  3.43623117e-01  3.61078933e-01
  3.78534749e-01  3.95990566e-01  4.13446382e-01  4.30902198e-01
  4.48358014e-01  4.83269

KeyboardInterrupt: 

In [12]:
predictions = model.predict(X_test)
accuracy = np.sum(predictions == y_test) / len(y_test)
print(f"AdaBoost Accuracy: {accuracy * 100:.2f}%")

AdaBoost Accuracy: 97.35%
